# Building an Artificial Neural Network (ANN) using Keras
## Introduction
### What is an Artificial Neural Network?
### Key Components:


## Step 1: Import Required Libraries


In [ ]:
# Import essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Keras and TensorFlow
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Import scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set random seeds for reproducibility
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

# Configure plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

## Step 2: Load and Explore the Dataset
### About the Dataset:


In [ ]:
# Load the dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=column_names)

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Info:")
print(df.info())

print("\nStatistical Summary:")
print(df.describe())

print("\nTarget Distribution:")
print(df['Outcome'].value_counts())
print(f"\nClass Balance: {df['Outcome'].value_counts(normalize=True)}")

## Step 3: Data Preprocessing
### Why Preprocessing is Important:
### StandardScaler Explanation:


In [ ]:
# Separate features and target
X = df.drop('Outcome', axis=1).values  # Features (8 columns)
y = df['Outcome'].values  # Target (binary: 0 or 1)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Split data into training and testing sets
# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])

# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on training data
X_test_scaled = scaler.transform(X_test)  # Transform test data using training statistics

print("\nFeature scaling completed!")
print("Mean of scaled training data:", X_train_scaled.mean(axis=0).round(2))
print("Std of scaled training data:", X_train_scaled.std(axis=0).round(2))

## Step 4: Build the ANN Model Architecture
### Model Architecture Concepts:
#### 1. **Sequential Model**
#### 2. **Dense Layer (Fully Connected Layer)**
#### 3. **Activation Functions**
#### 4. **Dropout Layer**
#### 5. **Batch Normalization**
### Our Architecture:


In [ ]:
# Create the Sequential model
model = Sequential(name='Diabetes_ANN')

# Input Layer + First Hidden Layer
# 16 neurons, ReLU activation
model.add(Dense(units=16, activation='relu', input_dim=8, name='input_layer'))
model.add(BatchNormalization())  # Normalize activations
model.add(Dropout(0.3))  # Drop 30% of neurons randomly

# Second Hidden Layer
# 8 neurons, ReLU activation
model.add(Dense(units=8, activation='relu', name='hidden_layer_1'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

# Third Hidden Layer
# 4 neurons, ReLU activation
model.add(Dense(units=4, activation='relu', name='hidden_layer_2'))
model.add(Dropout(0.2))

# Output Layer
# 1 neuron with sigmoid activation for binary classification
model.add(Dense(units=1, activation='sigmoid', name='output_layer'))

# Display model architecture
print("Model Architecture:")
print("="*70)
model.summary()
print("="*70)

## Step 5: Compile the Model
### Compilation Concepts:
#### 1. **Optimizer: Adam**
#### 2. **Loss Function: Binary Crossentropy**
#### 3. **Metrics: Accuracy**


In [ ]:
# Define the optimizer with custom learning rate
optimizer = Adam(learning_rate=0.001)

# Compile the model
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',  # For binary classification
    metrics=['accuracy']  # Track accuracy during training
)

print("Model compiled successfully!")
print("\nOptimizer: Adam (learning_rate=0.001)")
print("Loss Function: Binary Crossentropy")
print("Metrics: Accuracy")

## Step 6: Define Callbacks
### Callback Concepts:
#### 1. **EarlyStopping**
#### 2. **ModelCheckpoint**


In [ ]:
# Early Stopping: Stop training when validation loss doesn't improve
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=20,  # Wait 20 epochs before stopping
    restore_best_weights=True,  # Restore weights from best epoch
    verbose=1
)

# Model Checkpoint: Save the best model during training
checkpoint = ModelCheckpoint(
    filepath='best_diabetes_model.h5',
    monitor='val_accuracy',  # Monitor validation accuracy
    save_best_only=True,  # Only save if better than previous
    mode='max',  # Maximum accuracy is better
    verbose=1
)

print("Callbacks configured:")
print("1. EarlyStopping - Stops if val_loss doesn't improve for 20 epochs")
print("2. ModelCheckpoint - Saves best model based on val_accuracy")

## Step 7: Train the Model
### Training Concepts:
#### 1. **Epochs**
#### 2. **Batch Size**
#### 3. **Validation Split**
#### 4. **Training Process:**


In [ ]:
# Train the model
print("Starting model training...\n")

history = model.fit(
    X_train_scaled,  # Scaled training features
    y_train,  # Training labels
    epochs=150,  # Maximum number of epochs
    batch_size=32,  # Process 32 samples at a time
    validation_split=0.2,  # Use 20% of training data for validation
    callbacks=[early_stopping, checkpoint],  # Apply callbacks
    verbose=1  # Show progress bar
)

print("\n" + "="*70)
print("Training completed!")
print("="*70)

## Step 8: Visualize Training History
### Understanding Training Curves:
#### 1. **Loss Curves**
#### 2. **Accuracy Curves**
#### 3. **Ideal Scenario:**


In [ ]:
# Extract training history
train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs_range = range(1, len(train_loss) + 1)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot Loss
axes[0].plot(epochs_range, train_loss, 'b-', label='Training Loss', linewidth=2)
axes[0].plot(epochs_range, val_loss, 'r-', label='Validation Loss', linewidth=2)
axes[0].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (Binary Crossentropy)', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot Accuracy
axes[1].plot(epochs_range, train_acc, 'b-', label='Training Accuracy', linewidth=2)
axes[1].plot(epochs_range, val_acc, 'r-', label='Validation Accuracy', linewidth=2)
axes[1].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total epochs trained: {len(train_loss)}")
print(f"Final training loss: {train_loss[-1]:.4f}")
print(f"Final validation loss: {val_loss[-1]:.4f}")
print(f"Final training accuracy: {train_acc[-1]:.4f}")
print(f"Final validation accuracy: {val_acc[-1]:.4f}")

## Step 9: Evaluate on Test Data
### Evaluation Metrics:
#### 1. **Confusion Matrix**
#### 2. **Classification Metrics:**


In [ ]:
# Evaluate on test set
print("Evaluating model on test data...\n")
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print("="*70)
print("TEST SET RESULTS")
print("="*70)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print("="*70)

## Step 10: Make Predictions and Analyze Results

In [ ]:
# Make predictions on test set
y_pred_prob = model.predict(X_test_scaled)  # Get probabilities
y_pred = (y_pred_prob > 0.5).astype(int).flatten()  # Convert to binary (0 or 1)

print("Sample predictions:")
print("Actual | Predicted | Probability")
print("-" * 40)
for i in range(10):
    print(f"  {y_test[i]}    |     {y_pred[i]}     |   {y_pred_prob[i][0]:.4f}")

## Step 11: Confusion Matrix Visualization

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['No Diabetes (0)', 'Diabetes (1)'],
            yticklabels=['No Diabetes (0)', 'Diabetes (1)'],
            annot_kws={'size': 16, 'weight': 'bold'})
plt.title('Confusion Matrix - Diabetes Prediction', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Extract values
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix Breakdown:")
print(f"True Negatives (TN): {tn} - Correctly predicted no diabetes")
print(f"False Positives (FP): {fp} - Incorrectly predicted diabetes")
print(f"False Negatives (FN): {fn} - Missed diabetes cases")
print(f"True Positives (TP): {tp} - Correctly predicted diabetes")

## Step 12: Detailed Classification Report

In [ ]:
# Generate classification report
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred, 
                           target_names=['No Diabetes', 'Diabetes'],
                           digits=4))
print("="*70)

# Calculate additional metrics
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\nKey Metrics Summary:")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {precision:.4f} (Of predicted diabetes cases, {precision*100:.2f}% were correct)")
print(f"Recall:    {recall:.4f} (Caught {recall*100:.2f}% of actual diabetes cases)")
print(f"F1-Score:  {f1:.4f} (Harmonic mean of precision and recall)")

## Step 13: Making Predictions on New Data
### How to Use the Model for Prediction:


In [ ]:
# Example: Predict for a new patient
new_patient = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50]])  # Example data

print("New Patient Data:")
print(f"Pregnancies: {new_patient[0][0]}")
print(f"Glucose: {new_patient[0][1]}")
print(f"Blood Pressure: {new_patient[0][2]}")
print(f"Skin Thickness: {new_patient[0][3]}")
print(f"Insulin: {new_patient[0][4]}")
print(f"BMI: {new_patient[0][5]}")
print(f"Diabetes Pedigree Function: {new_patient[0][6]}")
print(f"Age: {new_patient[0][7]}")

# Scale the new data using the same scaler
new_patient_scaled = scaler.transform(new_patient)

# Make prediction
prediction_prob = model.predict(new_patient_scaled)[0][0]
prediction_class = int(prediction_prob > 0.5)

print("\n" + "="*70)
print("PREDICTION RESULT")
print("="*70)
print(f"Probability of Diabetes: {prediction_prob:.4f} ({prediction_prob*100:.2f}%)")
print(f"Prediction: {'DIABETES DETECTED' if prediction_class == 1 else 'NO DIABETES'}")
print("="*70)

## Step 14: Save and Load the Model
### Model Persistence:


In [ ]:
# Save the complete model
model.save('diabetes_ann_model.h5')
print("Model saved as 'diabetes_ann_model.h5'")

# To load the model later:
# loaded_model = keras.models.load_model('diabetes_ann_model.h5')
# predictions = loaded_model.predict(new_data_scaled)

print("\nTo load this model in the future, use:")
print("loaded_model = keras.models.load_model('diabetes_ann_model.h5')")

## Summary and Key Takeaways
### What We Learned:
### Next Steps to Improve:
### Resources for Further Learning:
